# DwightGPT on Colab
Runtime → Change runtime type → **T4 GPU** (or better). The whole pipeline (data, pretraining, SFT, demo) takes about 1–1.5 hours on a free T4 with the `small` preset.

In [ ]:
!git clone https://github.com/Hersh96/tiny-instruct-gpt.git
%cd tiny-instruct-gpt
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data + tokenizer
Downloads ~600 MB of TinyStories and ~150 MB of TinyStoriesInstruct, trains the BPE tokenizer and writes token files to `data/`.

In [ ]:
!python scripts/prepare.py --pretrain_mb 600 --sft_mb 150 --vocab_size 8192

## 2. Pretraining

In [ ]:
!python train.py --stage pretrain --preset small --max_iters 5000

In [ ]:
!python sample.py --ckpt out/pretrain/ckpt.pt --prompt "Once upon a time, there was a" --num_samples 2

## 3. Supervised fine-tuning (instruction following)

In [ ]:
!python train.py --stage sft --init_from out/pretrain/ckpt.pt --max_iters 1500

In [ ]:
!python sample.py --ckpt out/sft/ckpt.pt --chat "Write a short story about a brave little turtle. Use the words river and friend. Dwight should be in the story."

## 4. Loss curves and benchmarks

In [ ]:
!python scripts/plot.py out/pretrain/log.csv out/sft/log.csv --out assets/loss.png
!python bench.py --preset small
from IPython.display import Image, Markdown, display
display(Image('assets/loss.png'))
display(Markdown(open('assets/benchmarks.md').read()))

## 5. Optional: ablations (≈5 short runs)

In [ ]:
!ITERS=1500 bash scripts/ablate.sh

## 6. Live demo
Prints a public `gradio.live` link you can share for 72 hours.

In [ ]:
!python app.py --share

## 7. Save your work
Download the checkpoints and plots (or copy them to Google Drive).

In [ ]:
!zip -r results.zip out/pretrain/ckpt.pt out/sft/ckpt.pt out/*/log.csv data/tokenizer.json assets/
from google.colab import files
files.download('results.zip')